# A2780 Adaptive Treatment Simulations

This notebook accompanies the artifact-driven HTML dashboard for reproducing fitted A2780 trajectories and comparing cisplatin schedules. The default model is resolved from staged ranking, status, inheritance, and parameter exports each time the report is built.

The dashboard is a research sandbox, not a clinical dosing tool. The default graph shows only experimentally observable A2780Naive, total A2780cis, and total population.

## Mechanistic State Variables

- `N(t)`: A2780Naive population
- `S(t)`, `T(t)`: internal latent states required by the selected treated-coculture equation
- `C(t) = S(t) + T(t)`: total A2780cis population

The simulator uses effective coculture load terms:

`L_N = N + alpha_NC * C`

`L_C = C + alpha_CN * N`

Treatment is specified by windows with start day, stop day, cisplatin dose, and refresh interval. The latent states are hidden unless advanced diagnostics are enabled and must not be interpreted as separately measured populations.

## Simulation Equations

`dN/dt = G_N(N, L_N) - d_N*N - M_N*A_N(t)*H_N(E)*N`

`dS/dt = G_C(C, L_C) * S/C - d_C*S - M_C*A_C(t)*H_CS(E)*S`

`dT/dt = rho_T * G_C(C, L_C) * T/C - d_C*T - M_C*A_C(t)*H_CT(E)*T`

The context multipliers, timing functions, Hill effects, and latent initial fraction come from the selected fitted artifacts. Reversible phenotype switching is off by default and available only as a clearly labeled advanced research scenario.

In [ ]:
from pathlib import Path
import shutil, subprocess
try:
    from IPython.display import IFrame, display
except ImportError:
    IFrame = display = None

package_root = Path('..').resolve()
julia = shutil.which('julia')
if julia:
    subprocess.run([julia, '--project=.', 'scripts/build_adaptive_simulator_report.jl'], cwd=package_root, check=True)
else:
    print('Julia was not on PATH; displaying the last generated artifact-driven report.')
report = Path('..') / 'outputs' / 'reports' / 'a2780_adaptive_treatment_simulations.html'
if display:
    display(IFrame(src=str(report), width='100%', height=900))
else:
    assert report.is_file(), f'Missing simulator report: {report}'
    print(report.resolve())

## Suggested Use

1. Start with the observed-like 50:50 initial mix.
2. Adjust the observable Naive/cis starting composition; use latent-state controls only for advanced diagnostics.
3. Use the original reference schedule when comparing against the original experiment; media/treatment was refreshed every 2 days.
4. Compare continuous, pulse, AT50, hysteresis, dose-modulation, and monitoring-aware threshold schedules.
5. Compare total and cis AUC, final cis, time to progression, probability of control, dose, exposure, time off treatment, cycles, and overshoot.

All results after day 14 are extrapolation and should be interpreted through the ensemble sensitivity band.